# Day 33 · 工具链完善与状态管理

**配套讲义**: [`days/day-33.md`](../days/day-33.md) ｜ **本地可跑，不需要 GPU**

把 mock 工具补成有真实业务逻辑的实现，加上**幂等键、事务边界、失败降级话术**；并把会话状态从 `agent.py` 里抽成一个独立模块。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w6.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys
print("python:", sys.version.split()[0])
for m in ("numpy", "PIL", "yaml", "pandas"):
    try:
        mod = __import__(m)
        print(f"  {m:7s} {getattr(mod, '__version__', 'ok')}")
    except ImportError:
        print(f"  {m:7s} ❌ 缺 → pip install {m}")
print("\n→ 本机没 GPU 不影响今天：今天只用纯 Python / numpy")

## 1. 幂等性：亲手验证

In [ ]:
import sys; sys.path.insert(0, "..")
from src.agent.tools import execute_tool

results = [execute_tool("start_return", {"order_id": "A1", "reason": "尺码不合适"})
           for _ in range(3)]
for i, r in enumerate(results, 1):
    print(f"第 {i} 次:", r.data if hasattr(r, "data") else r)
unique = {str(r.data if hasattr(r, "data") else r) for r in results}
print("\n不同结果数:", len(unique), "（应该是 1 —— 三次调用只创建一个退货单）")

## 2. 幂等键的设计（自己写一个）

In [ ]:
import hashlib, json

def idem_key(session_id, tool_name, args):
    """确定性幂等键：同样的会话 + 工具 + 参数 → 同一个键。"""
    payload = json.dumps([session_id, tool_name, sorted(args.items())],
                         ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()[:16]

k1 = idem_key("s1", "start_return", {"order_id": "A1", "reason": "尺码不合适"})
k2 = idem_key("s1", "start_return", {"order_id": "A1", "reason": "尺码不合适"})
k3 = idem_key("s2", "start_return", {"order_id": "A1", "reason": "尺码不合适"})
print("同会话同参数:", k1 == k2, "（必须 True）")
print("异会话同参数:", k1 == k3, "（必须 False —— 不同用户各退一份）")

## 3. 抽出 state.py 的验收

`python -c "from src.agent.state import SessionState"` 能通过，且不触发 torch 导入 ——
说明状态管理和模型推理**解耦**了。这是工程化的关键一步。

In [ ]:
state_check = """
抽出来的模块必须满足：
  [ ] 不 import torch
  [ ] 不 import transformers
  [ ] 能独立单元测试
  [ ] 能序列化成 JSON（为了存 Redis / DB）
"""
print(state_check)

## 验收清单

- [ ] 幂等性自检通过：三次写操作只产生一个业务实体
- [ ] 每个工具的失败分支都有**具体**的 `suggestion`（不是「请稍后再试」）
- [ ] `state.py` 能独立 import 并使用（不依赖 `agent.py`）
- [ ] 能说出「半成功状态」的一个具体例子和它的防范方式

**卡住了？** 回看 [`days/day-33.md`](../days/day-33.md) 第五节「容易踩的坑」。

> **明天**：`days/day-34.md` —— Agent 主循环（ReAct + 护栏）